# Banking Data Creation

This notebook creates realistic banking datasets and saves them to a SQLite database.

## Data Generated:
- **Customers** - 500 customers with demographics and financial profiles
- **Transactions** - 2000 transaction records
- **Loans** - 150 loan records

---
*Run all cells to create and persist the data.*

In [5]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import date, timedelta
import os

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Pandas: 2.3.0
NumPy: 2.3.0


## Create Customers Dataset

In [2]:
np.random.seed(42)

n_customers = 500
customer_ids = [f'CUST{str(i).zfill(6)}' for i in range(1, n_customers + 1)]

first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 
                'William', 'Elizabeth', 'David', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas']
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 
          'San Antonio', 'San Diego', 'Dallas', 'San Jose', 'Austin', 'Jacksonville']
account_types = ['Checking', 'Savings', 'Premium Checking', 'Business']
account_segment = ['Retail', 'Premium', 'Corporate', 'Private']

customers = pd.DataFrame({
    'customer_id': customer_ids,
    'first_name': np.random.choice(first_names, n_customers),
    'last_name': np.random.choice(last_names, n_customers),
    'age': np.random.randint(22, 75, n_customers),
    'city': np.random.choice(cities, n_customers),
    'account_type': np.random.choice(account_types, n_customers, p=[0.35, 0.30, 0.20, 0.15]),
    'segment': np.random.choice(account_segment, n_customers, p=[0.45, 0.25, 0.20, 0.10]),
    'credit_score': np.random.randint(580, 850, n_customers),
    'annual_income': np.random.randint(30000, 250000, n_customers),
    'years_as_customer': np.random.randint(1, 30, n_customers),
    'credit_limit': np.where(np.random.random(n_customers) > 0.3, 
                             np.random.randint(5000, 50000, n_customers), 0)
})

print(f"Customers created: {len(customers)} rows")
customers.head()

Customers created: 500 rows


,customer_id,first_name,last_name,age,city,account_type,segment,credit_score,annual_income,years_as_customer,credit_limit
0,CUST000001,Michael,Anderson,38,San Diego,Business,Premium,595,183512,15,46692
1,CUST000002,Patricia,Gonzalez,30,Phoenix,Checking,Retail,708,196277,4,14940
2,CUST000003,Richard,Jones,54,San Diego,Checking,Private,642,110811,28,8924
3,CUST000004,Joseph,Thomas,74,San Jose,Business,Premium,688,85103,12,33054
4,CUST000005,David,Miller,41,New York,Premium Checking,Corporate,600,231884,6,14783


## Create Transactions Dataset

In [7]:
n_transactions = 2000
transaction_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment', 'Refund']
transaction_categories = ['Payroll', 'Bill Payment', 'Shopping', 'Dining', 'Utilities', 
                          'Entertainment', 'Healthcare', 'Travel', 'ATM', 'Online Transfer']

end_date = date.today()
start_date = end_date - timedelta(days=365)
transaction_dates = [start_date + timedelta(days=np.random.randint(0, 365)) 
                     for _ in range(n_transactions)]

transactions = pd.DataFrame({
    'transaction_id': [f'TXN{str(i).zfill(8)}' for i in range(1, n_transactions + 1)],
    'customer_id': np.random.choice(customer_ids, n_transactions),
    'transaction_date': transaction_dates,
    'transaction_type': np.random.choice(transaction_types, n_transactions, p=[0.25, 0.20, 0.15, 0.30, 0.10]),
    'category': np.random.choice(transaction_categories, n_transactions),
    'amount': np.abs(np.random.randn(n_transactions) * 500 + 200),
})

transactions.loc[transactions['transaction_type'] == 'Deposit', 'amount'] = np.abs(np.random.randn((transactions['transaction_type'] == 'Deposit').sum()) * 800 + 1500)
transactions.loc[transactions['transaction_type'] == 'Withdrawal', 'amount'] = np.abs(np.random.randn((transactions['transaction_type'] == 'Withdrawal').sum()) * 200 + 150)

transactions['amount'] = transactions['amount'].round(2)

print(f"Transactions created: {len(transactions)} rows")
transactions.head()

Transactions created: 2000 rows


,transaction_id,customer_id,transaction_date,transaction_type,category,amount
0,TXN00000001,CUST000377,2025-11-14,Payment,Travel,354.28
1,TXN00000002,CUST000152,2025-12-24,Deposit,Travel,2048.31
2,TXN00000003,CUST000198,2025-06-04,Refund,Utilities,66.12
3,TXN00000004,CUST000273,2026-02-16,Deposit,Travel,931.94
4,TXN00000005,CUST000156,2025-08-13,Withdrawal,Online Transfer,232.80


## Create Loans Dataset

In [9]:
n_loans = 150
loan_statuses = ['Active', 'Paid Off', 'Default', 'Pending']
loan_types = ['Personal', 'Home Mortgage', 'Auto', 'Business', 'Student']

loans = pd.DataFrame({
    'loan_id': [f'LOAN{str(i).zfill(6)}' for i in range(1, n_loans + 1)],
    'customer_id': np.random.choice(customer_ids, n_loans),
    'loan_type': np.random.choice(loan_types, n_loans),
    'loan_amount': np.random.randint(5000, 250000, n_loans),
    'interest_rate': np.round(np.random.uniform(3.5, 18.0, n_loans), 2),
    'loan_term_months': np.random.choice([12, 24, 36, 48, 60, 120, 180, 360], n_loans),
    'monthly_payment': 0.0,
    'outstanding_balance': 0.0,
    'status': np.random.choice(loan_statuses, n_loans, p=[0.55, 0.25, 0.10, 0.10]),
    'start_date': [start_date + timedelta(days=np.random.randint(0, 730)) for _ in range(n_loans)]
})

for idx, row in loans.iterrows():
    principal = row['loan_amount']
    rate = row['interest_rate'] / 100 / 12
    n_payments = row['loan_term_months']
    if rate > 0:
        payment = principal * (rate * (1 + rate)**n_payments) / ((1 + rate)**n_payments - 1)
    else:
        payment = principal / n_payments
    loans.at[idx, 'monthly_payment'] = round(payment, 2)
    
    if row['status'] == 'Paid Off':
        loans.at[idx, 'outstanding_balance'] = 0
    elif row['status'] == 'Active':
        months_elapsed = (date.today() - row['start_date']).days // 30
        paid_ratio = min(months_elapsed / n_payments, 0.85)
        loans.at[idx, 'outstanding_balance'] = round(principal * (1 - paid_ratio), 2)
    elif row['status'] == 'Default':
        loans.at[idx, 'outstanding_balance'] = round(principal * np.random.uniform(0.5, 0.9), 2)
    else:
        loans.at[idx, 'outstanding_balance'] = 0

print(f"Loans created: {len(loans)} rows")
loans.head()

Loans created: 150 rows


,loan_id,customer_id,loan_type,loan_amount,interest_rate,loan_term_months,monthly_payment,outstanding_balance,status,start_date
0,LOAN000001,CUST000296,Home Mortgage,206275,15.65,24,10065.40,0.00,Paid Off,2026-02-11
1,LOAN000002,CUST000286,Auto,187178,11.39,180,2173.52,107908.41,Default,2026-11-11
2,LOAN000003,CUST000281,Business,229497,14.20,360,2755.62,232046.97,Active,2026-08-05
3,LOAN000004,CUST000007,Home Mortgage,96676,14.96,180,1350.42,76388.16,Default,2025-05-02
4,LOAN000005,CUST000092,Business,87506,10.94,36,2862.35,0.00,Paid Off,2025-10-26


## Save to SQLite Database

In [ ]:
db_path = 'content/data/banking_data.db'

if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Removed existing database: {db_path}")

conn = sqlite3.connect(db_path)

customers.to_sql('customers', conn, if_exists='replace', index=False)
transactions.to_sql('transactions', conn, if_exists='replace', index=False)
loans.to_sql('loans', conn, if_exists='replace', index=False)

conn.commit()

print(f"\nDatabase saved: {db_path}")
print(f"Tables created: {['customers', 'transactions', 'loans']}")

cursor = conn.cursor()
for table in ['customers', 'transactions', 'loans']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  - {table}: {count} rows")

conn.close()
print("\nData creation complete!")

## Verify Database Contents

In [ ]:
conn = sqlite3.connect('content/data/banking_data.db')

In [ ]:
print("=== CUSTOMERS TABLE ===")
pd.read_sql("SELECT * FROM customers LIMIT 3", conn)

In [19]:
print("=== TRANSACTIONS TABLE ===")
pd.read_sql("SELECT * FROM transactions LIMIT 3", conn)

=== TRANSACTIONS TABLE ===


,transaction_id,customer_id,transaction_date,transaction_type,category,amount
0,TXN00000001,CUST000377,2025-11-14,Payment,Travel,354.28
1,TXN00000002,CUST000152,2025-12-24,Deposit,Travel,2048.31
2,TXN00000003,CUST000198,2025-06-04,Refund,Utilities,66.12


In [20]:
print("=== LOANS TABLE ===")
pd.read_sql("SELECT * FROM loans LIMIT 3", conn)

=== LOANS TABLE ===


,loan_id,customer_id,loan_type,loan_amount,interest_rate,loan_term_months,monthly_payment,outstanding_balance,status,start_date
0,LOAN000001,CUST000296,Home Mortgage,206275,15.65,24,10065.40,0.00,Paid Off,2026-02-11
1,LOAN000002,CUST000286,Auto,187178,11.39,180,2173.52,107908.41,Default,2026-11-11
2,LOAN000003,CUST000281,Business,229497,14.20,360,2755.62,232046.97,Active,2026-08-05


In [21]:
conn.close()

---
**Next**: Open `banking_analytics_workshop.ipynb` to analyze this data!